# Lab: Đánh giá mô hình phân loại

## 1. Vì sao chỉ Accuracy không đủ?

Accuracy = tỷ lệ dự đoán đúng. Nghe đơn giản, nhưng tệ ở các bài **mất cân bằng**: nếu 95% dataset là lớp 0, một model luôn dự đoán "0" đạt 95% accuracy mà *không học được gì*.

Ví dụ y học: chẩn đoán ung thư trong 1000 người khoẻ + 10 người bệnh. Model luôn nói "khoẻ" → 99% accuracy, bỏ sót 100% bệnh nhân. **Tai hoạ.**

Vì vậy ta cần các chỉ số khác: **Precision, Recall, F1, ROC-AUC**.

## 2. Confusion Matrix — bảng nền tảng

Với phân loại nhị phân (positive = lớp "có" / negative = lớp "không"):

| | Dự đoán Positive | Dự đoán Negative |
|---|---|---|
| **Thực Positive** | TP (True Positive) | FN (False Negative) |
| **Thực Negative** | FP (False Positive) | TN (True Negative) |

**Lưu ý quan trọng**: `sklearn.metrics.confusion_matrix(y_true, y_pred)` trả về:
$$\begin{pmatrix}TN & FP \\ FN & TP\end{pmatrix}$$
Hàng = nhãn thật, cột = dự đoán. Đây là quy ước sklearn — nhiều tài liệu khác đặt TP góc trên trái, vậy nên LUÔN xác định layout trước khi đọc.

## 3. Bốn chỉ số chính

### Accuracy
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
Tỷ lệ đoán đúng tổng. Chỉ tin cậy khi class cân bằng.

### Precision (độ chính xác)
$$\text{Precision} = \frac{TP}{TP + FP}$$
Trong số các mẫu **được dự đoán** là positive, bao nhiêu thật sự là positive? Cao = ít báo động giả.

### Recall (độ thu hồi / sensitivity)
$$\text{Recall} = \frac{TP}{TP + FN}$$
Trong số các mẫu **thực sự** là positive, bao nhiêu được model bắt được? Cao = ít bỏ sót.

### F1-score
$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
Trung bình điều hoà của Precision và Recall — phạt nặng khi *một trong hai* thấp.

### Specificity
$$\text{Specificity} = \frac{TN}{TN + FP}$$
Tỷ lệ negative thực sự được nhận ra đúng. = 1 − False Positive Rate.

## 4. Precision-Recall trade-off

Hai chỉ số thường *đối kháng* nhau. Tăng ngưỡng quyết định:
- Model "khắt khe" hơn → ít dự đoán positive → Precision tăng, Recall giảm.
- Model "dễ dãi" hơn → dự đoán nhiều positive → Recall tăng, Precision giảm.

Tuỳ bài toán mà ưu tiên cái nào:
- **Lọc spam**: ưu tiên Precision (tránh đẩy mail thật vào spam).
- **Chẩn đoán ung thư**: ưu tiên Recall (tránh bỏ sót bệnh).
- **Cân bằng**: dùng F1.

## 5. ROC và AUC

ROC = Receiver Operating Characteristic. Vẽ TPR (= Recall) theo FPR (= 1 − Specificity) khi quét ngưỡng.

**AUC** (Area Under ROC) = diện tích dưới đường, $\in [0, 1]$:
- AUC = 1: model hoàn hảo.
- AUC = 0.5: đoán mò.
- AUC < 0.5: tệ hơn đoán mò — nhưng **có thể lật prediction để được > 0.5** (không phải bỏ).

## 6. Multiclass — averaging methods

Khi có $> 2$ lớp, Precision/Recall/F1 tính riêng cho từng lớp rồi gộp lại bằng một trong ba cách:

- **Macro**: trung bình cộng đơn giản. Mỗi lớp có cùng trọng số.
- **Weighted**: trung bình có trọng số theo số mẫu mỗi lớp.
- **Micro**: cộng tất cả TP, FP, FN trên mọi lớp rồi tính một lần. Bằng accuracy.

Khi class mất cân bằng và muốn quan tâm đều các class → dùng **macro**. Khi quan tâm tổng thể → dùng **weighted**.

# THỰC HÀNH 1: Tính từng chỉ số bằng tay rồi so với sklearn

Cho `y_true` và `y_pred` thật, ta tự tính TP/FP/TN/FN, rồi từ đó suy ra mọi chỉ số. Cuối cùng so với sklearn để verify.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, accuracy_score,
                              precision_score, recall_score, f1_score,
                              classification_report, roc_curve, auc)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_breast_cancer, load_iris

np.random.seed(42)

# Ví dụ tay: 10 mẫu, 1 = positive, 0 = negative
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 1, 0, 0, 1, 1, 0, 1, 1])

TP = int(((y_true == 1) & (y_pred == 1)).sum())
TN = int(((y_true == 0) & (y_pred == 0)).sum())
FP = int(((y_true == 0) & (y_pred == 1)).sum())
FN = int(((y_true == 1) & (y_pred == 0)).sum())
print(f'TP = {TP}, TN = {TN}, FP = {FP}, FN = {FN}')

acc       = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
spec      = TN / (TN + FP) if (TN + FP) > 0 else 0

print(f'\nTay:    acc={acc:.3f}  prec={precision:.3f}  rec={recall:.3f}  f1={f1:.3f}  spec={spec:.3f}')
print(f'sklearn:acc={accuracy_score(y_true, y_pred):.3f}  '
      f'prec={precision_score(y_true, y_pred):.3f}  '
      f'rec={recall_score(y_true, y_pred):.3f}  '
      f'f1={f1_score(y_true, y_pred):.3f}')

In [ ]:
# Confusion matrix sklearn
cm = confusion_matrix(y_true, y_pred)
print('Confusion matrix sklearn (rows=true, cols=predicted):')
print(cm)
print(f'\n→ TN = cm[0,0] = {cm[0,0]}')
print(f'→ FP = cm[0,1] = {cm[0,1]}')
print(f'→ FN = cm[1,0] = {cm[1,0]}')
print(f'→ TP = cm[1,1] = {cm[1,1]}')

# THỰC HÀNH 2: Vì sao Accuracy lừa khi mất cân bằng

Sinh dữ liệu mất cân bằng 95/5, train một model "ngốc" (luôn dự đoán lớp đa số), xem các chỉ số lệch tới đâu.

In [ ]:
# 950 negative, 50 positive
y_imb = np.array([0] * 950 + [1] * 50)
y_dummy = np.zeros_like(y_imb)   # luôn dự đoán 0

print(f'Accuracy:  {accuracy_score(y_imb, y_dummy)*100:.2f}%   ← cao một cách lừa dối')
print(f'Precision: {precision_score(y_imb, y_dummy, zero_division=0)*100:.2f}%')
print(f'Recall:    {recall_score(y_imb, y_dummy)*100:.2f}%')
print(f'F1:        {f1_score(y_imb, y_dummy)*100:.2f}%   ← lộ diện: 0 vì recall=0')

# THỰC HÀNH 3: Đánh giá KNN trên Breast Cancer

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# Scale rồi train KNN
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_s, y_train)
y_pred = knn.predict(X_test_s)
y_proba = knn.predict_proba(X_test_s)[:, 1]

print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
# Confusion matrix vẽ đẹp
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(data.target_names); ax.set_yticklabels(data.target_names)
ax.set_xlabel('Dự đoán'); ax.set_ylabel('Thật')
ax.set_title('Confusion matrix')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)
plt.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
# ROC + AUC
fpr, tpr, thresh = roc_curve(y_test, y_proba)
auc_val = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, linewidth=2, label=f'KNN (AUC = {auc_val:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Đoán mò (AUC = 0.5)')
plt.xlabel('FPR (1 − Specificity)'); plt.ylabel('TPR (Recall)')
plt.title('ROC curve — Breast Cancer')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

# THỰC HÀNH 4: Multiclass — Iris

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(scaler.fit_transform(X_train), y_train)
y_pred = knn.predict(scaler.transform(X_test))

# 3 cách average khác nhau
print('Classification report (mặc định macro & weighted):')
print(classification_report(y_test, y_pred, target_names=iris.target_names))
print(f'F1 macro:    {f1_score(y_test, y_pred, average="macro"):.4f}')
print(f'F1 weighted: {f1_score(y_test, y_pred, average="weighted"):.4f}')
print(f'F1 micro:    {f1_score(y_test, y_pred, average="micro"):.4f}   (= accuracy)')

## Tổng kết

1. **Confusion matrix** là gốc — mọi chỉ số đều suy ra từ TP/TN/FP/FN.
2. Khi **mất cân bằng**, accuracy lừa → dùng Precision/Recall/F1.
3. **Precision–Recall** là cặp đối kháng — chọn theo bài toán:
   - Spam filter: precision quan trọng.
   - Cancer screening: recall quan trọng.
4. **F1** cân bằng, **AUC** đánh giá toàn quét ngưỡng.
5. **Multiclass**: macro / weighted / micro — biết khi nào dùng cái nào.

# BÀI TẬP VỀ NHÀ

## Bài 1: Wine dataset
Dùng `from sklearn.datasets import load_wine`. Train một model bất kỳ (KNN, RF, hay SVM). Báo cáo:
1. Confusion matrix.
2. Classification report.
3. F1 macro vs weighted — sự khác biệt?
4. Lớp nào model làm tệ nhất? Tại sao?

## Bài 2: Threshold tuning
Train logistic regression trên Breast Cancer. Sweep ngưỡng từ 0.1 → 0.9 (bước 0.05). Vẽ Precision và Recall theo ngưỡng. Tại ngưỡng nào F1 đạt max?

*Gợi ý:* `proba > threshold` thay vì gọi `predict()`.

## Bài 3: Precision-Recall curve
PR curve thường nói nhiều hơn ROC khi class mất cân bằng. Vẽ PR curve cho Breast Cancer:
```python
from sklearn.metrics import precision_recall_curve, average_precision_score
p, r, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
```
Vẽ kèm tiêu đề ghi AP. So với AUC ROC.

## Bài 4: Cohen's Kappa và MCC
Tìm hiểu hai chỉ số nâng cao:
- **Cohen's Kappa**: so accuracy với accuracy ngẫu nhiên.
- **MCC** (Matthews Correlation Coefficient): chỉ số tốt nhất cho binary với class mất cân bằng.

Tính cả hai trên Breast Cancer (dùng KNN). So sánh với F1.

*Gợi ý:* `from sklearn.metrics import cohen_kappa_score, matthews_corrcoef`.

## Bài 5: Class imbalance handling
Sinh dữ liệu mất cân bằng 90/10 (1000 mẫu, 2 feature). Train KNN. So sánh:
1. Default (không xử lý imbalance).
2. Oversampling lớp ít với SMOTE: `pip install imbalanced-learn`, `from imblearn.over_sampling import SMOTE`.
3. Class weight: `KNeighborsClassifier` không có, nhưng `LogisticRegression(class_weight='balanced')` có.

Báo cáo F1 cho lớp ít trong từng trường hợp.